In [1]:
print("Hello world")

Hello world


### API-Based Models

In [ ]:
from openai import OpenAI

client = OpenAI()

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role":"system", "content":"You are a helpful assistant"},
        {"role":"user", "content":"explain RAG briefly"}
    ],
    temparature=0.2
)

print(resp.choices[0].message.content)

### Local model inference(HuggingFace)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "meta-llama/Llama-2-7b-chat-hf"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

inputs = tok("Explain LoRA", return_tensor="pt")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tok.decode(outputs[0]))

### Canonical RAG Flow

In [ ]:
# graphql

User Query
 → Query Rewriting
 → Retriever (BM25 + Vector)
 → Reranker
 →Prompt assembly
 →LLM

### Simple RAG(vector search)

In [ ]:
query_emb = embedder.encode(query)
results = vectordb.search(query_emb, top_k=5)

context = "\n".join([r.text for i in results])
prompt = f"Answer using context:\n{context}\n\nQuestion: {query}"

### Hybrid search(BM25 + Embeddings)

In [ ]:
bm25_hits = bm25.search(query)
vector_hits = vectordb.search(embed(query))

combined = merge_and_score(bm25_hits, vector_hits)

### Reranking (cross-encoder)

In [ ]:
from sentence_transformer import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
pairs = [(query, doc.text) for doc in candidates]
scores = reranker.predict(pairs)

### Vector DBs (pinecone, Weaviate, Qdrant)

### QDrant

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

client.upsert(
    collection_names="docs",
    points=[
        {"id":1, "vector":embedding, "payload":{"source": "pdf"}}
    ]
)

hits = client.search("docs", query_vector=embedding, limit=5)

###  Embedding models(Training and Evaluation)

In [ ]:
"""
common models
    - text-embedding-3-large
    - bge-large
    - e5-large
    - XLM-R for multilingual

Training idea
    loss = cosine_loss(
        embed(query),
        embed(positive_doc),
        embed(negative_doc)
    )

Evaluation
    - Recall@K
    - MRR
    - nDCG
"""

### Multimodal AI (OCR, CLIP, Vision)

#### OCR -> LLM

In [ ]:
import pytesseract
text = pytesseract.image_to_string("invoice.png")

#### CLIP (Text -> Image)

In [ ]:
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained(...)

inputs = processor(text=["a receipt"], images=image, return_tensors="pt")
outputs = model(**inputs)

### Fine-Tuning LoRA/QLoRA

#### LoRA Setup (PEFT)

In [ ]:
from  peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(base_model, config)

#### QLoRA

In [ ]:
load_in_4bit = True
bnb_4bit_quant_type="nf4"

### Production AI Apps(Python and Pytorch)

In [ ]:
'''
-Batched inference
-Timeouts
-Retries
-Caching
'''

In [ ]:
@lru_cache
def cached_embedding(text):
    return embed(text)

#### Langchain/LlamaIndex Orchestration

##### Langchain RAG

In [ ]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever()
)

qa("what is hallucination?")

##### Tools & Memory

In [ ]:
agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    memory=ConversationBufferMemory
)

#### Evaluation and Hallucination Reduction

In [ ]:
"""" 
-Grounded prompts
-Citation enforcements
-Refusal policies
-Answer verification
"""

In [ ]:
prompt = """ 
Answer ONLY from the context.
If not present, say "I dont know".
"""

In [ ]:
#Metrics

"""
Faithfullness
Context Recall
Excat Match
LLM-as-a-judge
"""


#### Agentic Workflows (autogen, crewai)